# Feedback Cluster View for Google Sheets

This notebook reads the merged AST/LLM CSV and produces **one Excel workbook (`.xlsx`)** with two sheets:

- **feedback_cluster_summary**
- **feedback_cluster_patterns**

This is safer for Google Sheets than exporting raw CSV because the feedback text often contains commas and line breaks.


In [2]:
from pathlib import Path
import pandas as pd
from collections import Counter

# Update this path if your merged file has a different name
INPUT_FILE = Path("analysis_output_with_llm_feedback.csv")
OUTPUT_FILE = Path("feedback_cluster_view_for_google_sheets.xlsx")

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Could not find input file: {INPUT_FILE.resolve()}")

df = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df):,} rows from {INPUT_FILE}")
print(df.columns.tolist())

import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])


Loaded 1,708 rows from analysis_output_with_llm_feedback.csv
['File Name', 'Class Name', 'Method Name', 'Kind', 'Line Numbers', 'Tags', 'LLM Generated Feedback', 'Metadata']



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


0

In [3]:
# Normalize column names just in case there are small naming differences
column_map = {c.lower().strip(): c for c in df.columns}

file_col = next((column_map[k] for k in column_map if "file" in k), None)
method_col = next((column_map[k] for k in column_map if "method" in k), None)
tags_col = next((column_map[k] for k in column_map if k == "tags"), None)
feedback_col = next((column_map[k] for k in column_map if "llm generated feedback" in k), None)

required = {
    "File Name": file_col,
    "Method Name": method_col,
    "Tags": tags_col,
    "LLM Generated Feedback": feedback_col,
}
missing = [name for name, col in required.items() if col is None]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Using columns:")
for name, col in required.items():
    print(f"  {name}: {col}")


Using columns:
  File Name: File Name
  Method Name: Method Name
  Tags: Tags
  LLM Generated Feedback: LLM Generated Feedback


In [4]:
def normalize_tags(tag_value):
    if pd.isna(tag_value):
        return tuple()
    parts = [p.strip() for p in str(tag_value).split(",") if p.strip()]
    return tuple(sorted(parts))

work = df.copy()
work["Normalized_Tag_Pattern"] = work[tags_col].apply(normalize_tags)
work = work[work[feedback_col].notna()].copy()
work = work[work[feedback_col].astype(str).str.strip() != ""].copy()

print(f"Rows with non-empty feedback: {len(work):,}")


Rows with non-empty feedback: 1,694


In [5]:
summary_rows = []
pattern_rows = []

grouped = work.groupby([method_col, feedback_col], dropna=False)

for (method_name, feedback_text), group in grouped:
    tag_sets = [set(x) for x in group["Normalized_Tag_Pattern"]]
    pattern_counter = Counter(group["Normalized_Tag_Pattern"])

    if tag_sets:
        common_tags = sorted(set.intersection(*tag_sets)) if len(tag_sets) > 1 else sorted(next(iter(tag_sets)))
        union_tags = sorted(set.union(*tag_sets)) if len(tag_sets) > 1 else sorted(next(iter(tag_sets)))
        variable_tags = sorted(set(union_tags) - set(common_tags))
    else:
        common_tags = []
        variable_tags = []

    dominant_pattern, dominant_count = pattern_counter.most_common(1)[0]

    summary_rows.append({
        "Method Name": method_name,
        "LLM Generated Feedback": feedback_text,
        "Count": len(group),
        "Common Tags": ", ".join(common_tags),
        "Variable Tags": ", ".join(variable_tags),
        "Dominant Tag Pattern": ", ".join(dominant_pattern),
        "Dominant Pattern Count": dominant_count,
        "Unique Tag Pattern Count": len(pattern_counter),
    })

    for pattern, count in pattern_counter.items():
        files = sorted(group.loc[group["Normalized_Tag_Pattern"] == pattern, file_col].astype(str).tolist())
        pattern_rows.append({
            "Method Name": method_name,
            "LLM Generated Feedback": feedback_text,
            "Tag Pattern": ", ".join(pattern),
            "Count": count,
            "Is Dominant": pattern == dominant_pattern,
            "Files": " | ".join(files),
        })

summary_df = pd.DataFrame(summary_rows).sort_values(
    ["Method Name", "Count", "Unique Tag Pattern Count"],
    ascending=[True, False, False]
).reset_index(drop=True)

patterns_df = pd.DataFrame(pattern_rows).sort_values(
    ["Method Name", "LLM Generated Feedback", "Count", "Is Dominant"],
    ascending=[True, True, False, False]
).reset_index(drop=True)

print("Summary preview:")
display(summary_df.head(10))
print("Pattern preview:")
display(patterns_df.head(10))


Summary preview:


,Method Name,LLM Generated Feedback,Count,Common Tags,Variable Tags,Dominant Tag Pattern,Dominant Pattern Count,Unique Tag Pattern Count
0,ArrayCollection,"The constructor explicitly sets `size = 0`, so...",120,sets_size_to_0,,sets_size_to_0,120,1
1,ArrayCollection,"The constructor explicitly sets `size = 0`, so...",1,,,,1,1
2,add,`add` writes the new element at index `size` a...,77,increment_size,"array_insert_uses_size, grow_called_without_ca...","array_insert_uses_size, grow_check_equal, incr...",51,8
3,add,`add` writes the new element at index `size` a...,17,increment_size,"array_insert_uses_size, grow_called_without_ca...","array_insert_uses_size, grow_check_equal, incr...",9,6
4,add,`add` writes the new element at index `size` a...,11,increment_size,"array_insert_uses_size, grow_check_equal, grow...","array_insert_uses_size, grow_check_equal, incr...",8,4
5,add,"`add` increments `size`, so the count is updat...",4,increment_size,"grow_called_without_capacity_check, grow_check...","grow_called_without_capacity_check, increment_...",2,3
6,add,`add` writes the new element at index `size` a...,4,increment_size,"array_insert_uses_size, grow_called_without_ca...","array_insert_uses_size, grow_check_equal, incr...",2,3
7,add,Incorrect: this `add` implementation does not ...,2,"grow_called_without_capacity_check, size_no_up...",,"grow_called_without_capacity_check, size_no_up...",2,1
8,add,"`add` increments `size`, so the count is updat...",2,"grow_check_equal, increment_size",,"grow_check_equal, increment_size",2,1
9,add,`add` writes the new element at index `size` a...,2,"array_insert_uses_size, grow_check_equal, incr...",,"array_insert_uses_size, grow_check_equal, incr...",2,1


Pattern preview:


,Method Name,LLM Generated Feedback,Tag Pattern,Count,Is Dominant,Files
0,ArrayCollection,"The constructor explicitly sets `size = 0`, so...",sets_size_to_0,120,True,AfricanPenguin.java | AfricanWildDog.java | Am...
1,ArrayCollection,"The constructor explicitly sets `size = 0`, so...",,1,True,Salamander.java
2,add,Incorrect: this `add` implementation does not ...,"grow_called_without_capacity_check, size_no_up...",2,True,Crustacea.java | SiberianTiger.java
3,add,"`add` increments `size`, so the count is updat...","grow_missing, increment_size",1,True,Possum.java
4,add,"`add` increments `size`, so the count is updat...","grow_check_equal, increment_size",2,True,Bullfrog.java | Dormouse.java
5,add,"`add` increments `size`, so the count is updat...","grow_called_without_capacity_check, increment_...",2,True,HighlandCattle.java | Starfish.java
6,add,"`add` increments `size`, so the count is updat...","grow_check_equal, increment_size",1,False,Moorhen.java
7,add,"`add` increments `size`, so the count is updat...","growth_check_with_size_function, increment_size",1,False,Newt.java
8,add,`add` writes the new element at index `size` a...,"array_insert_uses_size, grow_check_equal, incr...",2,True,Ostrich.java | Prawn.java
9,add,`add` writes the new element at index `size` a...,"grow_called_without_capacity_check, increment_...",1,False,Dolphin.java


In [6]:
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="feedback_cluster_summary", index=False)
    patterns_df.to_excel(writer, sheet_name="feedback_cluster_patterns", index=False)

print(f"Saved workbook: {OUTPUT_FILE.resolve()}")
print("Upload this .xlsx file into Google Sheets or import its sheets into your existing workbook.")


Saved workbook: /Users/khalidmihlar/Code/ast-project/feedback_cluster_view_for_google_sheets.xlsx
Upload this .xlsx file into Google Sheets or import its sheets into your existing workbook.
